# Frozen V4 75게임 → 신규 50게임 temporal holdout

75게임에서 이미 학습된 V4 scaler와 coefficient를 다시 학습하지 않고 이후 50게임에 적용한다.

읽는 순서: checkpoint 무결성 → 125게임 dataset → gameId 차집합 → frozen scoring → physical-class ranking → 난이도별 metric → 한계와 full-run gate.

이번 결과는 Human placement 모방 평가다. Production 통합이나 game-over 회피 안전성 증거가 아니다.


## 1. 질문과 고정 계약

**왜:** 시간상 이후 데이터에서 고정 모델의 모방 안정성을 확인한다.

**입력:** immutable 75게임 checkpoint와 누적 125게임 dataset.

**계산:** feature, scaler, coefficient, ranking을 고정하고 평가 cohort만 신규 50게임으로 바꾼다.

**읽는 법:** smoke는 첫 normal/hard 각 1게임만 경로를 검증하며 성능 결론에 쓰지 않는다.

**보류 조건:** checksum, cohort count, leakage, selected label 또는 finite score 계약이 깨지면 즉시 중단한다.


In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import platform
import statistics
import tempfile
from collections import defaultdict
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import display

EXPERIMENT_ID = "placement-v4-frozen-75g-temporal-holdout-50g"
DATASET_FILES = ["manifest.json", "train.jsonl", "validation.jsonl", "rejected.jsonl"]
EXPECTED_SOURCE_LINES = 17485
EXPECTED_SOURCE_SHA256 = "6ab15deed40cedfcb094795aeffaef9003b4ce87694324063b81fed201615ec5"
EXPECTED_CHECKPOINT_SHA256 = "2ea7fe659220279c786d8c9c8e6f7ab6af70f970e0db985d6067b7a095c0ad53"
EXPECTED_BASE_SHA256 = {
    "manifest.json": "dbf1dd5f30177abc3b44d7bdcc7e21aaf2f8a3d0015a96ba6bdfd9506c71487a",
    "train.jsonl": "0a71b121514016721d50bda5e01b298002fe8820e0a110f6b62945b4f1550120",
    "validation.jsonl": "65511767072d46484a5e4e15821bde2ee91957d02b1e16af35ec5e9da2c5274d",
    "rejected.jsonl": "84c93b3b1d8e4d82eb4dcf094bab28c35d8834dedf9566bfbe72b008b47b1085",
}
EXPECTED_CUMULATIVE_SHA256 = {
    "manifest.json": "07fa9b76e868ac10f3cf4893aa7200ba0808971979f7f796152f7b3bfb3bd6ef",
    "train.jsonl": "986b5a6214aad13f13a239e4fe4a7c81582383dea6b16b1be6f046f2dbadd436",
    "validation.jsonl": "c468e24bbc48d94af5860bd2e5c7b90f2366a9cb9a32dba3abe7cdcce7af67bb",
    "rejected.jsonl": "0a19fab244c167384fdd5584f7495987ba4833576664fb78c1feacdb14d120e0",
}
FEATURE_NAMES = [
    "candidateResult.linesCleared", "candidateResult.gameOver",
    "metricDelta.aggregateHeight", "metricDelta.maxHeight",
    "metricDelta.holes", "metricDelta.bumpiness",
    *[f"metricDelta.columnHeights[{index}]" for index in range(10)],
]
FORBIDDEN_FEATURES = {
    "selectedBySource", "metadata.isWinner", "winnerPlayerId",
    "heuristicBaseline.score", "metadata.inputSequence",
}
RANKING_DIRECTION = "higher_linear_score_first"
CANONICAL_TIE_BREAK = "unicode_codepoint_equivalence_key"


## 2. Source, dataset과 checkpoint checksum

**왜:** 평가보다 먼저 immutable input 경계를 확인한다.

**입력:** live log, 75게임 dataset, 125게임 dataset, frozen checkpoint.

**계산:** line 수와 SHA-256, manifest 정책, checkpoint feature/scaler/coefficient를 검증한다.

**읽는 법:** 모든 checksum이 고정값과 일치해야 다음 단계로 진행한다.

**보류 조건:** 하나라도 다르면 새 경계를 임의로 채택하지 않는다.


In [ ]:
def find_repository_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "package.json").is_file() and (candidate / "python_tests").is_dir():
            return candidate
    raise AssertionError("REPOSITORY_ROOT_NOT_FOUND")


def resolve_path(value: str | Path, root: Path) -> Path:
    path = Path(value)
    return path if path.is_absolute() else root / path


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def read_json(path: Path) -> dict[str, Any]:
    value = json.loads(path.read_text(encoding="utf-8"))
    assert isinstance(value, dict), f"JSON_OBJECT_REQUIRED:{path.name}"
    return value


ROOT = find_repository_root(Path.cwd())
SOURCE_LOG = resolve_path(os.environ.get(
    "PLACEMENT_V4_FROZEN_SOURCE_LOG",
    ROOT.parent / "Tetris_multi/data/logs/game-events.jsonl",
), ROOT)
BASE_DATASET = resolve_path(os.environ.get(
    "PLACEMENT_V4_FROZEN_BASE_DATASET",
    ROOT / "data/datasets/human-behavior-v2-temporal-cumulative-75g-20260718",
), ROOT)
CUMULATIVE_DATASET = resolve_path(os.environ.get(
    "PLACEMENT_V4_FROZEN_CUMULATIVE_DATASET",
    ROOT / "data/datasets/human-behavior-v2-temporal-cumulative-125g-20260720",
), ROOT)
CHECKPOINT_PATH = resolve_path(os.environ.get(
    "PLACEMENT_V4_FROZEN_CHECKPOINT",
    ROOT / "data/ml-experiments/placement-v4-cumulative-75g-development-fit/full-report.json",
), ROOT)
for path in [SOURCE_LOG, CHECKPOINT_PATH, *[BASE_DATASET / name for name in DATASET_FILES], *[CUMULATIVE_DATASET / name for name in DATASET_FILES]]:
    assert path.is_file(), f"INPUT_REQUIRED:{path.name}"

SOURCE_SHA256_BEFORE = sha256_file(SOURCE_LOG)
SOURCE_LINE_COUNT_BEFORE = sum(1 for _ in SOURCE_LOG.open("rb"))
BASE_SHA256 = {name: sha256_file(BASE_DATASET / name) for name in DATASET_FILES}
CUMULATIVE_SHA256 = {name: sha256_file(CUMULATIVE_DATASET / name) for name in DATASET_FILES}
CHECKPOINT_SHA256 = sha256_file(CHECKPOINT_PATH)
assert SOURCE_LINE_COUNT_BEFORE == EXPECTED_SOURCE_LINES
assert SOURCE_SHA256_BEFORE == EXPECTED_SOURCE_SHA256
assert BASE_SHA256 == EXPECTED_BASE_SHA256
assert CUMULATIVE_SHA256 == EXPECTED_CUMULATIVE_SHA256
assert CHECKPOINT_SHA256 == EXPECTED_CHECKPOINT_SHA256

BASE_MANIFEST = read_json(BASE_DATASET / "manifest.json")
CUMULATIVE_MANIFEST = read_json(CUMULATIVE_DATASET / "manifest.json")
CHECKPOINT = read_json(CHECKPOINT_PATH)
assert BASE_MANIFEST["selectionPolicy"] == CUMULATIVE_MANIFEST["selectionPolicy"] == "all-completed"
assert BASE_MANIFEST["sourceFilter"] == CUMULATIVE_MANIFEST["sourceFilter"] == "human"
assert CUMULATIVE_MANIFEST["inputSha256"] == SOURCE_SHA256_BEFORE
assert CUMULATIVE_MANIFEST["totalGameCount"] == 127
assert CUMULATIVE_MANIFEST["eligibleGameCount"] == 125
assert CUMULATIVE_MANIFEST["acceptedPlacementLogCount"] == 5682
assert CUMULATIVE_MANIFEST["sampleCountBySource"]["human"] == 192129
assert CUMULATIVE_MANIFEST["acceptedPlacementLogCountByAiDifficulty"] == {"normal": 4500, "hard": 1182, "unknown": 0}
assert CUMULATIVE_MANIFEST["jsonParseFailureCount"] == 0

feature_contract = CHECKPOINT["featureContract"]
model_checkpoint = CHECKPOINT["modelCheckpoint"]
assert CHECKPOINT["representation"]["name"] == "V4"
assert feature_contract["featureNames"] == FEATURE_NAMES
assert model_checkpoint["coefficientFeatureOrder"] == FEATURE_NAMES
assert set(feature_contract["forbiddenLeakageInputs"]) == FORBIDDEN_FEATURES
SCALER_MEAN = np.asarray(feature_contract["mean"], dtype=np.float64)
SCALER_SCALE = np.asarray(feature_contract["scale"], dtype=np.float64)
COEFFICIENTS = np.asarray(model_checkpoint["coefficients"], dtype=np.float64)
assert SCALER_MEAN.shape == SCALER_SCALE.shape == COEFFICIENTS.shape == (16,)
assert np.isfinite(SCALER_MEAN).all() and np.isfinite(SCALER_SCALE).all() and np.isfinite(COEFFICIENTS).all()
assert (SCALER_SCALE > 0).all()
display(pd.DataFrame([{
    "sourceLines": SOURCE_LINE_COUNT_BEFORE,
    "sourceSha256": SOURCE_SHA256_BEFORE,
    "checkpointSha256": CHECKPOINT_SHA256,
    "frozenFeatureCount": len(FEATURE_NAMES),
}]))


## 3. 75게임과 125게임의 gameId 차집합

**왜:** random split이 아니라 시간상 새로 추가된 game 전체를 held-out cohort로 사용한다.

**입력:** 두 dataset의 train/validation JSONL.

**계산:** Human row와 selected 계약을 확인하고 gameId 차집합을 만든다.

**읽는 법:** full cohort는 normal 26게임과 hard 24게임, 총 2,202 decision이다.

**보류 조건:** base/new가 겹치거나 예상 count가 다르면 평가하지 않는다.


In [ ]:
def codepoint_key(value: Any) -> tuple[int, ...]:
    return tuple(ord(character) for character in str(value))


def decision_key(row: dict[str, Any]) -> str:
    return f"{row['gameId']}:{row['playerId']}:{int(row['pieceIndex'])}"


def scan_dataset(dataset: Path) -> dict[str, dict[str, Any]]:
    games: dict[str, dict[str, Any]] = {}
    for file_name in ["train.jsonl", "validation.jsonl"]:
        with (dataset / file_name).open("r", encoding="utf-8") as handle:
            for line in handle:
                row = json.loads(line)
                assert row["datasetSchemaVersion"] == 2
                assert row["source"] == "human"
                assert type(row["selectedBySource"]) is bool
                game = games.setdefault(row["gameId"], {
                    "difficulty": row["aiDifficulty"],
                    "candidateCount": 0,
                    "selectedCounts": defaultdict(int),
                })
                assert game["difficulty"] == row["aiDifficulty"]
                game["candidateCount"] += 1
                game["selectedCounts"][decision_key(row)] += int(row["selectedBySource"])
    for game_id, game in games.items():
        assert all(count == 1 for count in game["selectedCounts"].values()), f"SELECTED_COUNT_MISMATCH:{game_id}"
        game["decisionCount"] = len(game["selectedCounts"])
    return games


BASE_GAMES = scan_dataset(BASE_DATASET)
CUMULATIVE_GAMES = scan_dataset(CUMULATIVE_DATASET)
BASE_GAME_IDS = set(BASE_GAMES)
CUMULATIVE_GAME_IDS = set(CUMULATIVE_GAMES)
NEW_GAME_IDS = sorted(CUMULATIVE_GAME_IDS - BASE_GAME_IDS, key=codepoint_key)
GAME_ID_LEAKAGE_COUNT = len(BASE_GAME_IDS & set(NEW_GAME_IDS))
assert len(BASE_GAME_IDS) == 75
assert len(CUMULATIVE_GAME_IDS) == 125
assert len(NEW_GAME_IDS) == 50
assert GAME_ID_LEAKAGE_COUNT == 0

NEW_GAME_IDS_BY_DIFFICULTY = {
    difficulty: [game_id for game_id in NEW_GAME_IDS if CUMULATIVE_GAMES[game_id]["difficulty"] == difficulty]
    for difficulty in ["normal", "hard"]
}
EXPECTED_NEW_COUNTS = {
    "normal": {"games": 26, "decisions": 1430, "candidates": 48323},
    "hard": {"games": 24, "decisions": 772, "candidates": 26001},
}
for difficulty, expected in EXPECTED_NEW_COUNTS.items():
    game_ids = NEW_GAME_IDS_BY_DIFFICULTY[difficulty]
    actual = {
        "games": len(game_ids),
        "decisions": sum(CUMULATIVE_GAMES[game_id]["decisionCount"] for game_id in game_ids),
        "candidates": sum(CUMULATIVE_GAMES[game_id]["candidateCount"] for game_id in game_ids),
    }
    assert actual == expected, f"NEW_COHORT_COUNT_MISMATCH:{difficulty}:{actual}"

RUN_MODE = os.environ.get("PLACEMENT_V4_FROZEN_RUN_MODE", "smoke")
assert RUN_MODE in {"smoke", "full"}, "RUN_MODE_MUST_BE_SMOKE_OR_FULL"
if RUN_MODE == "smoke":
    EVALUATED_GAME_IDS = [
        NEW_GAME_IDS_BY_DIFFICULTY["normal"][0],
        NEW_GAME_IDS_BY_DIFFICULTY["hard"][0],
    ]
else:
    EVALUATED_GAME_IDS = NEW_GAME_IDS
FULL_COHORT_COVERED = EVALUATED_GAME_IDS == NEW_GAME_IDS
POLICY_DECISION_ELIGIBLE = bool(FULL_COHORT_COVERED)
CANONICAL_RESULT_ELIGIBLE = False
display(pd.DataFrame([
    {"difficulty": difficulty, **EXPECTED_NEW_COUNTS[difficulty], "firstGameId": NEW_GAME_IDS_BY_DIFFICULTY[difficulty][0]}
    for difficulty in ["normal", "hard"]
]))


## 4. Frozen 16-feature와 V4 occupied-cell grouping

**왜:** 새 50게임으로 parameter를 다시 맞추지 않고 동일한 physical placement 단위를 평가한다.

**입력:** 선택된 신규 game candidate와 checkpoint mean/scale/coefficient.

**계산:** 필요한 필드만 streaming으로 읽고 pieceType과 sorted occupied cells로 label-free class를 만든다.

**읽는 법:** 같은 class 내부 feature가 같고 decision마다 selected class가 정확히 하나여야 한다.

**보류 조건:** feature 불일치, non-Human row, 복수 selected class 또는 leakage가 있으면 중단한다.


In [ ]:
def placement_key(row: dict[str, Any]) -> str:
    placement = row["candidatePlacement"]
    return f"{placement['pieceType']}:{int(placement['rotation'])}:{int(placement['row'])}:{int(placement['col'])}"


def equivalence_key(row: dict[str, Any]) -> str:
    placement = row["candidatePlacement"]
    cells = sorted((int(cell["row"]), int(cell["col"])) for cell in placement["cells"])
    assert len(cells) == 4 and len(set(cells)) == 4
    return f"{placement['pieceType']}|" + ";".join(f"{r},{c}" for r, c in cells)


def feature_vector(row: dict[str, Any]) -> np.ndarray:
    result, delta = row["candidateResult"], row["metricDelta"]
    values = np.asarray([
        int(result["linesCleared"]), int(bool(result["gameOver"])),
        float(delta["aggregateHeight"]), float(delta["maxHeight"]),
        float(delta["holes"]), float(delta["bumpiness"]),
        *[float(value) for value in delta["columnHeights"]],
    ], dtype=np.float64)
    assert values.shape == (16,) and np.isfinite(values).all()
    return values


def load_compact_candidates(dataset: Path, game_ids: set[str]) -> list[dict[str, Any]]:
    candidates: list[dict[str, Any]] = []
    for file_name in ["train.jsonl", "validation.jsonl"]:
        with (dataset / file_name).open("r", encoding="utf-8") as handle:
            for line in handle:
                row = json.loads(line)
                if row["gameId"] not in game_ids:
                    continue
                assert row["source"] == "human" and type(row["selectedBySource"]) is bool
                candidates.append({
                    "gameId": row["gameId"],
                    "decisionKey": decision_key(row),
                    "difficulty": row["aiDifficulty"],
                    "placementKey": placement_key(row),
                    "equivalenceKey": equivalence_key(row),
                    "selected": row["selectedBySource"],
                    "feature": feature_vector(row),
                    "linesCleared": int(row["candidateResult"]["linesCleared"]),
                    "gameOver": bool(row["candidateResult"]["gameOver"]),
                })
    return candidates


CANDIDATES = load_compact_candidates(CUMULATIVE_DATASET, set(EVALUATED_GAME_IDS))
DECISIONS: dict[str, list[dict[str, Any]]] = defaultdict(list)
for candidate in CANDIDATES:
    DECISIONS[candidate["decisionKey"]].append(candidate)
for key, candidates in DECISIONS.items():
    assert sum(candidate["selected"] for candidate in candidates) == 1, f"SELECTED_COUNT_MISMATCH:{key}"
    classes: dict[str, list[dict[str, Any]]] = defaultdict(list)
    for candidate in candidates:
        classes[candidate["equivalenceKey"]].append(candidate)
    assert sum(any(member["selected"] for member in members) for members in classes.values()) == 1
    for class_key, members in classes.items():
        assert all(np.array_equal(member["feature"], members[0]["feature"]) for member in members), f"EQUIVALENCE_FEATURE_MISMATCH:{class_key}"

expected_candidates = sum(CUMULATIVE_GAMES[game_id]["candidateCount"] for game_id in EVALUATED_GAME_IDS)
expected_decisions = sum(CUMULATIVE_GAMES[game_id]["decisionCount"] for game_id in EVALUATED_GAME_IDS)
assert len(CANDIDATES) == expected_candidates
assert len(DECISIONS) == expected_decisions
assert not set(EVALUATED_GAME_IDS) & BASE_GAME_IDS
display(pd.DataFrame([{
    "runMode": RUN_MODE,
    "evaluatedGames": len(EVALUATED_GAME_IDS),
    "evaluatedDecisions": len(DECISIONS),
    "evaluatedCandidates": len(CANDIDATES),
    "leakage": GAME_ID_LEAKAGE_COUNT,
}]))


## 5. Frozen score와 canonical physical-class ranking

**왜:** checkpoint가 새 game에서 Human이 선택한 physical class를 얼마나 높게 두는지 계산한다.

**입력:** compact candidate, frozen mean/scale/coefficient.

**계산:** feature를 고정 scaler로 transform하고 linear score가 높은 class부터 Unicode code-point로 동점을 푼다.

**읽는 법:** selectedRank 1이면 모델 첫 class와 Human 선택 class가 같다.

**보류 조건:** score가 finite가 아니거나 ranking에 selected class가 하나가 아니면 중단한다.


In [ ]:
def score_decision(key: str, candidates: list[dict[str, Any]]) -> dict[str, Any]:
    classes: dict[str, list[dict[str, Any]]] = defaultdict(list)
    for candidate in candidates:
        classes[candidate["equivalenceKey"]].append(candidate)
    rows = []
    for class_key in sorted(classes, key=codepoint_key):
        members = sorted(classes[class_key], key=lambda row: codepoint_key(row["placementKey"]))
        representative = members[0]
        scaled = (representative["feature"] - SCALER_MEAN) / SCALER_SCALE
        score = float(scaled @ COEFFICIENTS)
        assert np.isfinite(score)
        rows.append({
            "key": class_key,
            "score": score,
            "selected": any(member["selected"] for member in members),
            "linesCleared": representative["linesCleared"],
        })
    assert sum(row["selected"] for row in rows) == 1
    ranked = sorted(rows, key=lambda row: (-row["score"], codepoint_key(row["key"])))
    selected = next(row for row in rows if row["selected"])
    selected_rank = next(index for index, row in enumerate(ranked, 1) if row["key"] == selected["key"])
    top_score = ranked[0]["score"]
    raw_selected = next(candidate for candidate in candidates if candidate["selected"])
    duplicate = any(
        not candidate["selected"] and np.array_equal(candidate["feature"], raw_selected["feature"])
        for candidate in candidates
    )
    return {
        "gameId": candidates[0]["gameId"],
        "decisionKey": key,
        "difficulty": candidates[0]["difficulty"],
        "physicalClassCount": len(rows),
        "choiceRequired": len(rows) > 1,
        "selectedRank": selected_rank,
        "selectedAtTopScore": selected["score"] == top_score,
        "topScoreTieCount": sum(row["score"] == top_score for row in rows),
        "selectedFeatureDuplicate": duplicate,
        "lineClearOption": any(row["linesCleared"] > 0 for row in rows),
        "selectedScore": selected["score"],
    }


PREDICTIONS = [score_decision(key, DECISIONS[key]) for key in sorted(DECISIONS, key=codepoint_key)]
assert len(PREDICTIONS) == len(DECISIONS)
assert all(np.isfinite(row["selectedScore"]) for row in PREDICTIONS)
display(pd.DataFrame(PREDICTIONS[:5])[
    ["gameId", "decisionKey", "difficulty", "physicalClassCount", "selectedRank", "topScoreTieCount"]
])


## 6. Worked example

**왜:** 한 decision이 class grouping, frozen scaling과 ranking을 거치는 과정을 손으로 확인한다.

**입력:** 정렬상 첫 choice-required decision.

**계산:** 각 physical class 대표의 score와 selected 여부를 canonical 순위로 펼친다.

**읽는 법:** raw candidate 수보다 physical class 수가 작을 수 있고 selected class rank가 metric의 한 행이 된다.

**보류 조건:** 선택 class가 없거나 복수이면 전체 metric을 계산하지 않는다.


In [ ]:
WORKED_DECISION_KEY = next(
    key for key in sorted(DECISIONS, key=codepoint_key)
    if len({row["equivalenceKey"] for row in DECISIONS[key]}) > 1
)
worked_candidates = DECISIONS[WORKED_DECISION_KEY]
worked_classes: dict[str, list[dict[str, Any]]] = defaultdict(list)
for candidate in worked_candidates:
    worked_classes[candidate["equivalenceKey"]].append(candidate)
worked_rows = []
for class_key in sorted(worked_classes, key=codepoint_key):
    members = sorted(worked_classes[class_key], key=lambda row: codepoint_key(row["placementKey"]))
    scaled = (members[0]["feature"] - SCALER_MEAN) / SCALER_SCALE
    worked_rows.append({
        "classKey": class_key,
        "aliasCount": len(members),
        "selectedClass": any(member["selected"] for member in members),
        "score": float(scaled @ COEFFICIENTS),
    })
worked_rows = sorted(worked_rows, key=lambda row: (-row["score"], codepoint_key(row["classKey"])))
for rank, row in enumerate(worked_rows, 1):
    row["rank"] = rank
assert sum(row["selectedClass"] for row in worked_rows) == 1
display(pd.DataFrame([{
    "decisionKey": WORKED_DECISION_KEY,
    "rawCandidateCount": len(worked_candidates),
    "physicalClassCount": len(worked_rows),
    "selectedClassRank": next(row["rank"] for row in worked_rows if row["selectedClass"]),
}]))
display(pd.DataFrame(worked_rows).head(10))


## 7. 전체·normal·hard metric과 진단

**왜:** 전체 평균만으로 난이도와 game별 편차를 숨기지 않는다.

**입력:** choice-required physical-class prediction.

**계산:** Top-1, Top-3, MRR, mean rank와 game macro/median/min/max, worst game, tie·duplicate·line-clear segment를 집계한다.

**읽는 법:** primary는 choiceRequired 결과다. Smoke 숫자는 코드 경로 확인용일 뿐 성능 결론이 아니다.

**보류 조건:** normal 또는 hard segment가 비거나 metric이 finite가 아니면 full 판단을 보류한다.


In [ ]:
def summarize(rows: list[dict[str, Any]]) -> dict[str, Any]:
    if not rows:
        return {
            "decisionCount": 0, "top1Count": 0, "top3Count": 0,
            "top1Accuracy": None, "top3Accuracy": None,
            "mrr": None, "meanSelectedRank": None,
            "topScoreTieDecisionCount": 0,
        }
    ranks = np.asarray([row["selectedRank"] for row in rows], dtype=np.int64)
    summary = {
        "decisionCount": len(rows),
        "top1Count": int((ranks == 1).sum()),
        "top3Count": int((ranks <= 3).sum()),
        "top1Accuracy": float(np.mean(ranks == 1)),
        "top3Accuracy": float(np.mean(ranks <= 3)),
        "mrr": float(np.mean(1.0 / ranks)),
        "meanSelectedRank": float(np.mean(ranks)),
        "topScoreTieDecisionCount": sum(row["topScoreTieCount"] > 1 for row in rows),
    }
    assert all(value is None or not isinstance(value, float) or np.isfinite(value) for value in summary.values())
    return summary


def evaluate_scope(rows: list[dict[str, Any]]) -> dict[str, Any]:
    choice = [row for row in rows if row["choiceRequired"]]
    primary = summarize(choice)
    assert primary["decisionCount"] > 0
    per_game = []
    for game_id in sorted({row["gameId"] for row in rows}, key=codepoint_key):
        game_choice = [row for row in choice if row["gameId"] == game_id]
        per_game.append({"gameId": game_id, **summarize(game_choice)})
    game_mrr = [row["mrr"] for row in per_game]
    assert all(value is not None and np.isfinite(value) for value in game_mrr)
    worst = min(per_game, key=lambda row: (row["mrr"], codepoint_key(row["gameId"])))
    return {
        "choiceRequired": primary,
        "allDecisions": summarize(rows),
        "gameDistribution": {
            "gameCount": len(per_game),
            "macroMrr": float(statistics.fmean(game_mrr)),
            "medianMrr": float(statistics.median(game_mrr)),
            "minimumMrr": float(min(game_mrr)),
            "maximumMrr": float(max(game_mrr)),
            "worstGame": worst,
        },
        "perGame": per_game,
    }


EVALUATION = {
    "all": evaluate_scope(PREDICTIONS),
    "normal": evaluate_scope([row for row in PREDICTIONS if row["difficulty"] == "normal"]),
    "hard": evaluate_scope([row for row in PREDICTIONS if row["difficulty"] == "hard"]),
}
LINE_CLEAR_SEGMENT = summarize([
    row for row in PREDICTIONS if row["choiceRequired"] and row["lineClearOption"]
])
GAME_OVER_CANDIDATE_COUNT = sum(candidate["gameOver"] for candidate in CANDIDATES)
GAME_OVER_DECISION_COUNT = len({
    candidate["decisionKey"] for candidate in CANDIDATES if candidate["gameOver"]
})
assert GAME_OVER_CANDIDATE_COUNT == GAME_OVER_DECISION_COUNT == 0
display(pd.DataFrame([
    {"scope": scope, **EVALUATION[scope]["choiceRequired"]}
    for scope in ["all", "normal", "hard"]
]))


## 8. Rejection과 game-over 안전성 한계

**왜:** exporter가 제외한 top-out placement를 평가 row로 몰래 복구하지 않았음을 명시한다.

**입력:** cumulative rejected.jsonl과 사전 등록된 세 game/piece.

**계산:** 세 rejection이 SELECTED_PLACEMENT_NOT_REACHABLE로 존재하는지만 확인한다.

**읽는 법:** accepted candidate의 gameOver=true가 0이므로 game-over 회피 안전성을 검증하지 않는다.

**보류 조건:** runtime hard safety는 이 Notebook의 metric으로 승인하지 않는다.


In [ ]:
EXPECTED_TOP_OUT_REJECTIONS = [
    {"gameId": "1692219B-1784370584589", "pieceIndex": 64, "difficulty": "normal"},
    {"gameId": "3C3114C6-1784430567426", "pieceIndex": 18, "difficulty": "hard"},
    {"gameId": "50487EB2-1784466429659", "pieceIndex": 36, "difficulty": "hard"},
]
rejected_keys = set()
with (CUMULATIVE_DATASET / "rejected.jsonl").open("r", encoding="utf-8") as handle:
    for line in handle:
        row = json.loads(line)
        rejected_keys.add((row.get("gameId"), row.get("pieceIndex"), row.get("reason")))
for expected in EXPECTED_TOP_OUT_REJECTIONS:
    assert (
        expected["gameId"], expected["pieceIndex"], "SELECTED_PLACEMENT_NOT_REACHABLE"
    ) in rejected_keys
GAME_OVER_LIMITATIONS = [
    "accepted cohort has zero gameOver=true candidates",
    "this experiment evaluates Human imitation ranking only",
    "game-over avoidance safety is not validated",
    "runtime hard safety remains separate future work",
]
display(pd.DataFrame(EXPECTED_TOP_OUT_REJECTIONS))


## 9. Source 재검증과 결정적 report

**왜:** 평가가 source, dataset 또는 checkpoint를 변경하지 않았음을 확인한다.

**입력:** 실행 전 checksum과 계산 결과.

**계산:** 실행 후 bytes를 다시 hash하고 timestamp 없는 sorted JSON report를 atomic replace한다.

**읽는 법:** smoke는 awaiting_user_full_run이며 세 eligibility flag가 승격을 막는다.

**보류 조건:** source mutation, output collision 또는 non-finite JSON이면 report를 남기지 않는다.


In [ ]:
SOURCE_SHA256_AFTER = sha256_file(SOURCE_LOG)
SOURCE_LINE_COUNT_AFTER = sum(1 for _ in SOURCE_LOG.open("rb"))
BASE_SHA256_AFTER = {name: sha256_file(BASE_DATASET / name) for name in DATASET_FILES}
CUMULATIVE_SHA256_AFTER = {name: sha256_file(CUMULATIVE_DATASET / name) for name in DATASET_FILES}
CHECKPOINT_SHA256_AFTER = sha256_file(CHECKPOINT_PATH)
assert SOURCE_SHA256_AFTER == SOURCE_SHA256_BEFORE
assert SOURCE_LINE_COUNT_AFTER == SOURCE_LINE_COUNT_BEFORE
assert BASE_SHA256_AFTER == BASE_SHA256
assert CUMULATIVE_SHA256_AFTER == CUMULATIVE_SHA256
assert CHECKPOINT_SHA256_AFTER == CHECKPOINT_SHA256

STATUS = "needs_policy_decision" if FULL_COHORT_COVERED else "awaiting_user_full_run"
REPORT = {
    "experimentId": EXPERIMENT_ID,
    "status": STATUS,
    "runMode": RUN_MODE,
    "fullCohortCovered": FULL_COHORT_COVERED,
    "policyDecisionEligible": POLICY_DECISION_ELIGIBLE,
    "canonicalResultEligible": CANONICAL_RESULT_ELIGIBLE,
    "heldOutEvidence": bool(FULL_COHORT_COVERED),
    "productionIntegration": False,
    "frozenCheckpoint": True,
    "parameterFitCount": 0,
    "source": {
        "lineCountBefore": SOURCE_LINE_COUNT_BEFORE,
        "lineCountAfter": SOURCE_LINE_COUNT_AFTER,
        "sha256Before": SOURCE_SHA256_BEFORE,
        "sha256After": SOURCE_SHA256_AFTER,
    },
    "datasets": {
        "baseChecksumsBefore": BASE_SHA256,
        "baseChecksumsAfter": BASE_SHA256_AFTER,
        "cumulativeChecksumsBefore": CUMULATIVE_SHA256,
        "cumulativeChecksumsAfter": CUMULATIVE_SHA256_AFTER,
    },
    "checkpoint": {
        "sha256Before": CHECKPOINT_SHA256,
        "sha256After": CHECKPOINT_SHA256_AFTER,
        "featureNames": FEATURE_NAMES,
        "mean": SCALER_MEAN.tolist(),
        "scale": SCALER_SCALE.tolist(),
        "coefficients": COEFFICIENTS.tolist(),
    },
    "rankingContract": {
        "direction": RANKING_DIRECTION,
        "canonicalTieBreak": CANONICAL_TIE_BREAK,
        "evaluationUnit": "occupied-cell physical class",
        "choiceRequired": True,
    },
    "cohorts": {
        "baseGameCount": len(BASE_GAME_IDS),
        "cumulativeGameCount": len(CUMULATIVE_GAME_IDS),
        "newGameCount": len(NEW_GAME_IDS),
        "newDecisionCount": sum(CUMULATIVE_GAMES[game_id]["decisionCount"] for game_id in NEW_GAME_IDS),
        "newCandidateCount": sum(CUMULATIVE_GAMES[game_id]["candidateCount"] for game_id in NEW_GAME_IDS),
        "byDifficulty": EXPECTED_NEW_COUNTS,
        "evaluatedGameIds": EVALUATED_GAME_IDS,
        "evaluatedGameCount": len(EVALUATED_GAME_IDS),
        "evaluatedDecisionCount": len(DECISIONS),
        "evaluatedCandidateCount": len(CANDIDATES),
        "gameIdLeakageCount": GAME_ID_LEAKAGE_COUNT,
    },
    "evaluation": EVALUATION,
    "diagnostics": {
        "topScoreTieDecisionCount": sum(row["topScoreTieCount"] > 1 for row in PREDICTIONS),
        "selectedFeatureDuplicateDecisionCount": sum(row["selectedFeatureDuplicate"] for row in PREDICTIONS),
        "lineClearOptionSegment": LINE_CLEAR_SEGMENT,
        "gameOverCandidateCount": GAME_OVER_CANDIDATE_COUNT,
        "gameOverDecisionCount": GAME_OVER_DECISION_COUNT,
    },
    "workedExample": {
        "decisionKey": WORKED_DECISION_KEY,
        "rawCandidateCount": len(worked_candidates),
        "physicalClassCount": len(worked_rows),
        "selectedClassRank": next(row["rank"] for row in worked_rows if row["selectedClass"]),
    },
    "topOutRejections": EXPECTED_TOP_OUT_REJECTIONS,
    "limitations": GAME_OVER_LIMITATIONS,
    "environment": {
        "python": platform.python_version(),
        "platform": platform.system(),
        "architecture": platform.machine(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
    },
}
REPORT_TEXT = json.dumps(REPORT, ensure_ascii=False, sort_keys=True, indent=2, allow_nan=False) + "\n"


def same_inode(left: Path, right: Path) -> bool:
    left_stat, right_stat = left.stat(), right.stat()
    return (left_stat.st_dev, left_stat.st_ino) == (right_stat.st_dev, right_stat.st_ino)


def atomic_write_text(output_path: Path, text: str, protected_sources: list[Path]) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    for source in protected_sources:
        assert output_path.resolve(strict=False) != source.resolve(strict=True), "REPORT_SOURCE_COLLISION"
        if os.path.lexists(output_path):
            assert not output_path.is_symlink(), "REPORT_OUTPUT_SYMLINK_NOT_ALLOWED"
            assert not same_inode(output_path, source), "REPORT_OUTPUT_INODE_COLLISION"
    descriptor, staging_name = tempfile.mkstemp(prefix=f".{output_path.name}.", suffix=".tmp", dir=output_path.parent)
    staging_path = Path(staging_name)
    try:
        with os.fdopen(descriptor, "w", encoding="utf-8", newline="\n") as handle:
            handle.write(text)
            handle.flush()
            os.fsync(handle.fileno())
        assert not staging_path.is_symlink()
        os.replace(staging_path, output_path)
    finally:
        if staging_path.exists():
            staging_path.unlink()


REPORT_PATH = resolve_path(os.environ.get(
    "PLACEMENT_V4_FROZEN_REPORT",
    "/private/tmp/placement-v4-frozen-75g-temporal-holdout-50g-smoke.json",
), ROOT)
PROTECTED_SOURCES = [
    SOURCE_LOG, CHECKPOINT_PATH,
    *[BASE_DATASET / name for name in DATASET_FILES],
    *[CUMULATIVE_DATASET / name for name in DATASET_FILES],
]
atomic_write_text(REPORT_PATH, REPORT_TEXT, PROTECTED_SOURCES)
REPORT_SHA256 = sha256_file(REPORT_PATH)
assert REPORT_SHA256 == hashlib.sha256(REPORT_TEXT.encode("utf-8")).hexdigest()
display(pd.DataFrame([{
    "status": STATUS,
    "runMode": RUN_MODE,
    "reportSha256": REPORT_SHA256,
    "sourceUnchanged": SOURCE_SHA256_AFTER == SOURCE_SHA256_BEFORE,
}]))


## 10. 결과를 읽는 법과 user full-run gate

1. evaluation.all.choiceRequired에서 전체 held-out metric을 본다.
2. normal과 hard를 나란히 보고 난이도 차이를 확인한다.
3. gameDistribution.worstGame과 tie·duplicate·line-clear 진단을 확인한다.
4. accepted cohort에 game-over candidate가 없다는 안전성 한계를 유지한다.

Smoke는 경로 검증일 뿐이다. Full 실행은 사용자가 다음 명령을 직접 수행한다.

    PLACEMENT_V4_FROZEN_RUN_MODE=full \
    PLACEMENT_V4_FROZEN_REPORT=/private/tmp/placement-v4-frozen-75g-temporal-holdout-50g-full.json \
    conda run -n tetris-ml python -m jupyter nbconvert \
      --to notebook \
      --execute scripts/ml/notebooks/placement_v4_frozen_75g_temporal_holdout_50g.ipynb \
      --output placement-v4-frozen-75g-temporal-holdout-50g-executed-full.ipynb \
      --output-dir /private/tmp \
      --ExecutePreprocessor.timeout=1200

Full report도 canonicalResultEligible=false, productionIntegration=false이며 자동 채택하지 않는다.
